In [1]:
%pip install pytorch_forecasting

  Using cached pytorch_forecasting-1.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached lightning-2.5.6-py3-none-any.whl.metadata (42 kB)
  Using cached lightning_utilities-0.15.2-py3-none-any.whl.metadata (5.7 kB)
  Using cached torchmetrics-1.8.2-py3-none-any.whl.metadata (22 kB)
  Using cached pytorch_lightning-2.5.6-py3-none-any.whl.metadata (20 kB)
  Using cached aiohttp-3.13.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.meta

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import optuna
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import mean_squared_error
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import NaNLabelEncoder
from pytorch_lightning import Trainer
from sklearn.metrics import mean_squared_error, r2_score

/opt/venvs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
dataset_path = 'data_train_post_n_smoothing.csv'
df = pd.read_csv(dataset_path, sep="\t")
df.shape

(1000, 701)

In [6]:
X = df.iloc[:, :500]
y = df['y']

# converter wide -> long format (TSFresh input)
df_long = X.copy()
df_long['id'] = df_long.index

df_long = df_long.melt(
    id_vars='id',
    var_name='time',
    value_name='value'
)

df_long = df_long.merge(
    df[['y']].reset_index().rename(columns={'index': 'id'}),
    on='id',
    how='left'
)
df_long = df_long.rename(columns={'time': 'time_idx'})
df_long["time_idx"] = df_long["time_idx"].astype(int)
df_long

,id,time_idx,value,y
0,0,0,1.000000,0.29
1,1,0,1.000000,0.30
2,2,0,1.000000,0.11
3,3,0,1.000000,0.61
4,4,0,1.000000,0.74
...,...,...,...,...
499995,995,499,0.029409,0.33
499996,996,499,0.017936,0.29
499997,997,499,0.026152,0.36
499998,998,499,0.013301,0.69


In [7]:

# # Assuming df_long is as above
# max_encoder_length = 500  # full sequence length
# max_prediction_length = 1 # we only predict one value

# For TFT, the target must be time-varying.
# But your 'y' is static — so we'll create a dummy target = value itself
df_long["target"] = df_long["value"]

training = TimeSeriesDataSet(
    df_long,
    time_idx="time_idx",
    target="target",  # <- dummy target (since 'y' is static)
    group_ids=["id"],
    max_encoder_length=500,
    max_prediction_length=1,  # minimal
    static_reals=["y"],       # real static variable to predict from
    time_varying_known_reals=["value"],
)

train_loader = training.to_dataloader(train=True, batch_size=32, num_workers=4)

# Model
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    loss=torch.nn.MSELoss(),
    output_size=1,
    hidden_continuous_size=16,
)

trainer = Trainer(max_epochs=50, gpus=1)
trainer.fit(tft, train_loader)

/opt/venvs/torch/lib/python3.10/site-packages/pytorch_forecasting/data/timeseries/_timeseries.py:1850: UserWarning: Min encoder length and/or min_prediction_idx and/or min prediction length and/or lags are too large for 1000 series/groups which therefore are not present in the dataset index. This means no predictions can be made for those series. First 10 removed groups: [{'__group_id__id': 0}, {'__group_id__id': 1}, {'__group_id__id': 2}, {'__group_id__id': 3}, {'__group_id__id': 4}, {'__group_id__id': 5}, {'__group_id__id': 6}, {'__group_id__id': 7}, {'__group_id__id': 8}, {'__group_id__id': 9}]
  warnings.warn(


AssertionError: filters should not remove entries all entries - check encoder/decoder lengths and lags

**Estrutura da Rede**

O modelo proposto tem 3 blocos:

1. Embedding temporal (1D linear)
Transforma cada ponto do sinal em um vetor de dimensão d_model.

2. Transformer Encoder
Captura dependências entre posições do sinal (autocorrelação, padrões locais, etc.) via self-attention.

3. Pooling + MLP
Reduz a sequência a um vetor fixo e aplica camadas densas para prever y.

In [8]:
class TransformerRegressor(nn.Module):
    def __init__(self, input_length=500, d_model=64, nhead=4, num_layers=3, dim_feedforward=128, dropout=0.1):
        super().__init__()
        
        # 1. Projeção linear de 1D -> d_model
        self.input_proj = nn.Linear(1, d_model)
        
        # 2. Codificador Transformer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # [batch, seq, features]
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 3. Pooling + regressão final
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # Entrada: [batch, seq_length]
        x = x.unsqueeze(-1)  # [batch, seq, 1]
        x = self.input_proj(x)  # [batch, seq, d_model]
        x = self.transformer(x)  # [batch, seq, d_model]
        x = x.mean(dim=1)  # média temporal
        return self.fc(x)

In [10]:
import torch.optim as optim
import numpy as np

# Preparar dados
X = df.iloc[:, :500].to_numpy(dtype=np.float32)
y = df['y'].to_numpy(dtype=np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Tensores
X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

# Modelo
model = TransformerRegressor(input_length=500)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Treino
for epoch in range(40):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: Loss={loss.item():.4f}")


Epoch 0: Loss=0.6019
Epoch 5: Loss=0.0864
Epoch 10: Loss=0.0778
Epoch 15: Loss=0.0535
Epoch 20: Loss=0.0527
Epoch 25: Loss=0.0532
Epoch 30: Loss=0.0493
Epoch 35: Loss=0.0496


In [11]:
# Avaliação
model.eval()
with torch.no_grad():
    y_pred = model(X_test_t).numpy()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"RMSE: {rmse:.4f}, R²: {r2:.3f}")

RMSE: 0.2283, R²: 0.055


In [ ]:
#### setup training with optuna

In [ ]:

from tqdm import tqdm
import optuna


# ============================================================
# Transformer Regressor
# ============================================================
class TransformerRegressor(nn.Module):
    def __init__(self, input_length=500, d_model=64, nhead=4, num_layers=3,
                 dim_feedforward=128, dropout=0.1):
        super().__init__()
        
        self.input_proj = nn.Linear(1, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.fc = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)        # [batch, seq, 1]
        x = self.input_proj(x)     # [batch, seq, d_model]
        x = self.transformer(x)    # [batch, seq, d_model]
        x = x.mean(dim=1)          # pooling temporal
        return self.fc(x)

# ============================================================
# Função objetivo do Optuna
# ============================================================
def objective(trial):

    # 🔹 Hiperparâmetros a otimizar
    d_model = trial.suggest_categorical("d_model", [32, 64, 128])
    nhead = trial.suggest_categorical("nhead", [2, 4, 8])
    num_layers = trial.suggest_int("num_layers", 1, 4)
    dim_feedforward = trial.suggest_categorical("dim_feedforward", [64, 128, 256, 512])
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    # 🔹 Dataloader
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size, shuffle=True
    )

    # 🔹 Modelo, loss e optimizer
    model = TransformerRegressor(
        input_length=X_train.shape[1],
        d_model=d_model,
        nhead=nhead,
        num_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout
    )

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 🔹 Treinamento
    n_epochs = 25
    for epoch in tqdm(range(n_epochs), desc=f"Trial {trial.number}", leave=False):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

    # 🔹 Validação
    model.eval()
    with torch.no_grad():
        preds = model(X_test_t).numpy()
        y_true = y_test_t.numpy()
        rmse = np.sqrt(mean_squared_error(y_true, preds))

    return rmse

# ============================================================
# Execução da otimização
# ============================================================
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=18, show_progress_bar=True)

print("\nBest trial:")
trial = study.best_trial
print(f"  RMSE: {trial.value:.4f}")
print("  Params:", trial.params)

[I 2025-11-13 16:48:54,777] A new study created in memory with name: no-name-f04fa7b7-c350-463e-bc88-2ebada2b83be
Best trial: 0. Best value: 0.0987419:   6%|▌         | 1/18 [24:24<6:54:50, 1464.13s/it]

[I 2025-11-13 17:13:18,911] Trial 0 finished with value: 0.09874194783114437 and parameters: {'d_model': 32, 'nhead': 8, 'num_layers': 2, 'dim_feedforward': 512, 'dropout': 0.3721118434996955, 'lr': 0.00019161905481056727, 'batch_size': 16}. Best is trial 0 with value: 0.09874194783114437.



Best trial: 0. Best value: 0.0987419:  11%|█         | 2/18 [31:39<3:48:59, 858.69s/it] 

[I 2025-11-13 17:20:33,790] Trial 1 finished with value: 0.09996335393765811 and parameters: {'d_model': 64, 'nhead': 2, 'num_layers': 3, 'dim_feedforward': 64, 'dropout': 0.2701587402112925, 'lr': 0.0006771920318313136, 'batch_size': 16}. Best is trial 0 with value: 0.09874194783114437.



Best trial: 0. Best value: 0.0987419:  17%|█▋        | 3/18 [1:08:48<6:11:12, 1484.84s/it]

...


Best trial: 5. Best value: 0.0951649:  61%|██████    | 11/18 [3:29:32<1:45:46, 906.67s/it] 

[I 2025-11-13 20:18:26,816] Trial 10 finished with value: 0.0972181171613063 and parameters: {'d_model': 64, 'nhead': 4, 'num_layers': 1, 'dim_feedforward': 128, 'dropout': 0.18524799094215877, 'lr': 0.006589069234971458, 'batch_size': 64}. Best is trial 5 with value: 0.09516490213399405.



Best trial: 5. Best value: 0.0951649:  67%|██████▋   | 12/18 [3:35:45<1:14:26, 744.47s/it]

[I 2025-11-13 20:24:40,310] Trial 11 finished with value: 0.10156932679487249 and parameters: {'d_model': 64, 'nhead': 4, 'num_layers': 1, 'dim_feedforward': 128, 'dropout': 0.15656152106820553, 'lr': 0.007633207964193371, 'batch_size': 64}. Best is trial 5 with value: 0.09516490213399405.



Trial 12:   4%|▍         | 1/25 [00:42<17:04, 42.71s/it]

In [ ]:
class SimpleTFT(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, n_heads=4):
        super().__init__()
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=n_heads,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=3)
        self.input_proj = nn.Linear(input_size, hidden_size)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.input_proj(x)
        enc = self.encoder(x)
        out = enc.mean(dim=1)  # global average pooling
        return self.fc(out)